# Notebook 03 — sNaïve Baselines

## Purpose

Produce three reference forecasts ("baselines") for each of the two forecasting tasks, using simple deterministic rules with no learnable parameters. These baselines serve two purposes in the project:

1. **Methodological floor.** Any machine learning model in the downstream notebooks must beat these baselines to justify its complexity. If a LightGBM model can't outperform "predict last week, same hour," then the ML pipeline has a problem.
2. **Diagnostic instrument.** Each baseline's signed-error pattern reveals where structural complexity lives in the data. The previous-year baseline will systematically underpredict in FWES and NOTH (the structural-growth zones from notebook 01). The previous-week baseline will fail on holidays. The historical-average baseline will fail on extreme events. These predictable failures provide evidence-of-need for the features and models we add later.

This notebook is structurally simple: no training, no hyperparameter search, no feature matrix. Each baseline is a deterministic rule applied directly to the historical `pd` values.

## The three baselines

The three baselines are taken directly from the assignment's "Hints / Suggested Approach" section:

| Baseline | Rule | Captures |
|---|---|---|
| `prev_week` | Predict `pd` at target hour t = `pd` at hour `t - 168h` (7 days earlier) | Weekly cycle (weekday vs weekend pattern) |
| `prev_year` | Predict `pd` at target hour t = `pd` at hour `t - 8760h` (365 days earlier) | Annual cycle (winter vs summer demand) |
| `hist_avg` | Predict `pd` at target hour t = average `pd` for this bus at this (hour-of-day, day-of-week) combination, computed over 2022-2024 training data | Joint hour × weekday climatology |

## Task-specific admissibility

The two forecasting tasks have different data-availability constraints, which affects which baselines are admissible.

**Next-day task** (forecast issued day D-1, predicting day D). All three baselines are directly admissible:
- `prev_week` lookback at `t - 168h` is always ≥ 7 days before forecast_created_at ✓
- `prev_year` lookback at `t - 8760h` is always ≥ 1 year before forecast_created_at ✓
- `hist_avg` uses only training-period data (2022-2024) ✓

**Next-month task** (forecast issued day 1 of month M-1, predicting all of month M). The `prev_week` rule is **not directly admissible** for most target hours, because `t - 168h` would land inside the forecasted month itself for target hours past day 7 of M. To preserve the "recent-historical lookup" spirit of the weekly baseline while respecting the leakage constraint, we substitute:

| Original | Substitute | Justification |
|---|---|---|
| `prev_week` (t - 168h) | `prev_60d` (t - 1440h) | The shortest lag that is consistently admissible across all hours of the forecasted month is approximately 60 days (the gap from forecast_created_at to the last hour of month M is up to ~60 days). The `prev_60d` baseline preserves the same-hour-and-same-day-of-week lookup pattern of `prev_week` at a longer horizon. |

The other two baselines remain admissible for the next-month task without modification.

## Cold-start handling

From notebook 01, **1,725 buses appear only in 2025** with no training-period history. For these buses:

- `prev_week` returns NaN for the first 7 days of the bus's appearance
- `prev_year` returns NaN for the entire 2025 test period (no prior-year data exists)
- `hist_avg` returns NaN entirely (no training-period observations to average)

We apply a **zone-average fallback** to handle these cases. For a cold-start bus in zone Z at hour t, when the primary baseline rule returns NaN, we predict the average pd across all non-cold-start buses in zone Z at hour t. This uses the zone-membership information we do have about each cold-start bus to produce a defensible non-zero prediction.

The fallback is conservative — it does not pretend to know the cold-start bus's individual load level, only its zone's typical level. For the 11% of test rows affected, this is methodologically the right tradeoff: a reasonable approximation is better than a forced NaN that breaks evaluation, and better than a guess (zero or global mean) that ignores zone structure.

The fallback is applied uniformly across all three baselines and both tasks.

## Outputs

Six forecast files written to `data/processed/forecasts/`, each in the assignment's required output schema:

| File | Task | Baseline | Rows |
|---|---|---|---|
| `forecast_prev_week_nextday.parquet` | Next-day | prev_week (t - 168h) | ~37M |
| `forecast_prev_year_nextday.parquet` | Next-day | prev_year (t - 8760h) | ~37M |
| `forecast_hist_avg_nextday.parquet` | Next-day | hist_avg (bus × hour × dow climatology) | ~37M |
| `forecast_prev_60d_nextmonth.parquet` | Next-month | prev_60d (t - 1440h, substituted from prev_week) | ~37M |
| `forecast_prev_year_nextmonth.parquet` | Next-month | prev_year (t - 8760h) | ~37M |
| `forecast_hist_avg_nextmonth.parquet` | Next-month | hist_avg | ~37M |

Each file covers 4,208 buses × all hours of 2025 = approximately 37 million rows.

## Implementation strategy

We load the historical bus data once (the same 132M-row filtered DataFrame from notebook 02, before feature engineering — just bus_unique_id, timestamp, pd, zone_name). For each baseline:

1. Compute the rule on the 2025 target hours using the appropriate lookback
2. Apply the zone-average fallback to rows where the rule returned NaN
3. Format into the required output schema with `model_name`, `forecast_created_at`, etc.
4. Write to parquet

We do **not** reload the raw 320M-row dataset. We can either:
- Load directly from notebook 02's feature parquet files (already filtered to the 4,208 buses, with pd and timestamp present)
- Re-stream the raw bus files filtered to the forecastable universe

The first option is faster and uses what we already built. We'll use that.

## Runtime estimate

End-to-end: approximately 10-15 minutes. The main costs are:
- Loading the per-year feature parquet files (~1-2 minutes)
- Computing the historical average lookup table (~30 seconds)
- Per-baseline timestamp merges (~1-2 minutes each)
- Writing the 6 output files (~2-3 minutes)

In [1]:
"""
Imports, configuration, and path setup for notebook 03.

This cell establishes the runtime environment for baseline forecast generation:
  - Standard library: pathlib for portable paths, warnings to suppress benign
    pandas FutureWarnings, gc and time for memory management and runtime
    instrumentation, psutil for monitoring available RAM.
  - Numeric/data stack: numpy, pandas, pyarrow. We use pyarrow.parquet
    directly for column-selective reads when loading the feature files
    from notebook 02.

Path conventions: this notebook lives in assignment2/notebooks/. Data paths
are relative to that location. Inputs come from data/processed/features/
(notebook 02 outputs) and data/processed/audit/ (notebook 01 outputs).
Forecast outputs land in data/processed/forecasts/ which we create if it
does not yet exist.
"""

# Standard library
from pathlib import Path
import warnings
import gc
import time
import psutil

# Numeric and data
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Display and warning configuration
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
warnings.simplefilter("ignore", category=FutureWarning)

# Paths (relative to notebook location: assignment2/notebooks/)
DATA_DIR = Path("../data")
AUDIT_DIR = Path("../data/processed/audit")
FEATURES_DIR = Path("../data/processed/features")
FORECASTS_DIR = Path("../data/processed/forecasts")
FORECASTS_DIR.mkdir(parents=True, exist_ok=True)

# Per-year feature file paths from notebook 02
YEARS = [2022, 2023, 2024, 2025]
NEXTDAY_FEATURE_FILES = {y: FEATURES_DIR / f"features_nextday_{y}.parquet" for y in YEARS}
NEXTMONTH_FEATURE_FILES = {y: FEATURES_DIR / f"features_nextmonth_{y}.parquet" for y in YEARS}

# Audit artifact from notebook 01
FORECASTABLE_BUS_LIST_PATH = AUDIT_DIR / "forecastable_bus_list.parquet"

# Verify all expected inputs exist before proceeding
for y in YEARS:
    assert NEXTDAY_FEATURE_FILES[y].exists(), f"Missing: {NEXTDAY_FEATURE_FILES[y]}"
    assert NEXTMONTH_FEATURE_FILES[y].exists(), f"Missing: {NEXTMONTH_FEATURE_FILES[y]}"
assert FORECASTABLE_BUS_LIST_PATH.exists(), (
    f"Missing audit artifact: {FORECASTABLE_BUS_LIST_PATH}. Run notebook 01 first."
)

print(f"Feature files located in {FEATURES_DIR.resolve()}")
print(f"Audit artifacts located in {AUDIT_DIR.resolve()}")
print(f"Forecast outputs will be written to {FORECASTS_DIR.resolve()}")

Feature files located in /Users/gavinyu/Desktop/ECESIS Investments Assignments/ECESIS-2026-Summer-Power-Systems-Modeling-Assignment/assignment2/data/processed/features
Audit artifacts located in /Users/gavinyu/Desktop/ECESIS Investments Assignments/ECESIS-2026-Summer-Power-Systems-Modeling-Assignment/assignment2/data/processed/audit
Forecast outputs will be written to /Users/gavinyu/Desktop/ECESIS Investments Assignments/ECESIS-2026-Summer-Power-Systems-Modeling-Assignment/assignment2/data/processed/forecasts


In [2]:
"""
Load the historical pd data and bus-zone mapping needed for baselines.

We need three things in memory:

1. Historical pd data covering 2022-2025 with (bus_unique_id, timestamp, pd,
   zone_name). The 2022-2024 data is the "training period" used by hist_avg
   and as lookback for prev_year forecasting the early hours of 2025. The
   2025 data provides the set of (bus, timestamp) combinations we need to
   predict for, plus its own pd column serves as lookback for prev_week
   (week 2 onwards uses week 1's 2025 data as a 168h-ago lookback).

2. The forecastable bus list with each bus's most-common zone. This is the
   canonical bus → zone mapping used both for the output schema (zone_id)
   and for the cold-start fallback (zone-average lookup).

3. A target grid for 2025: the set of (bus, timestamp) combinations the
   baselines must produce predictions for. This is just the 2025 rows of
   the historical data filtered to 2025 timestamps.

We load only the columns we actually need from the feature files (just 4 of
the 30+ available columns), which keeps memory bounded.

Memory: peak ~3-4 GB. Each year's slice is ~800 MB after filtering to the
4 needed columns.
"""

t0 = time.time()

# Columns needed from the feature files
NEEDED_COLS = ["bus_unique_id", "timestamp", "pd", "zone_name"]

# Load all four years from the next-day feature files (the pd, timestamp,
# bus_unique_id, zone_name columns are identical across the next-day and
# next-month feature files, so it doesn't matter which we read from)
print("Loading historical pd data from feature files...")
year_dfs = []
for y in YEARS:
    table = pq.read_table(NEXTDAY_FEATURE_FILES[y], columns=NEEDED_COLS)
    df = table.to_pandas()
    del table
    year_dfs.append(df)
    print(f"  {y}: {len(df):>11,} rows")

historical = pd.concat(year_dfs, ignore_index=True)
del year_dfs
gc.collect()

print(f"\nCombined historical DataFrame: {historical.shape}")
print(f"Memory: {historical.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
print(f"Date range: {historical['timestamp'].min()} to {historical['timestamp'].max()}")
print(f"Unique buses: {historical['bus_unique_id'].nunique():,}")
print(f"Unique zones: {sorted(historical['zone_name'].unique())}")

# Load the forecastable bus list (bus → zone mapping)
forecastable_buses = pd.read_parquet(FORECASTABLE_BUS_LIST_PATH)
print(f"\nForecastable bus list: {len(forecastable_buses):,} buses × {forecastable_buses.shape[1]} columns")

# Identify cold-start buses: present in 2025 but not in 2022-2024
buses_in_train = set(historical[historical["timestamp"].dt.year < 2025]["bus_unique_id"].unique())
buses_in_test = set(historical[historical["timestamp"].dt.year == 2025]["bus_unique_id"].unique())
cold_start_buses = buses_in_test - buses_in_train
print(f"\nCold-start buses (2025 only, no training history): {len(cold_start_buses):,}")
print(f"Non-cold-start buses (have training history): {len(buses_in_test - cold_start_buses):,}")

# Sort by (bus, timestamp) — required for lag-style lookups later
historical = historical.sort_values(["bus_unique_id", "timestamp"], kind="stable").reset_index(drop=True)

# Sanity check on row count
expected_2025_rows = len(historical[historical["timestamp"].dt.year == 2025])
print(f"\n2025 rows (target prediction set): {expected_2025_rows:,}")

elapsed = time.time() - t0
print(f"\nLoad complete in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Loading historical pd data from feature files...
  2022:  32,692,332 rows
  2023:  32,897,778 rows
  2024:  32,962,294 rows
  2025:  32,427,554 rows

Combined historical DataFrame: (130979958, 4)
Memory: 2.32 GB
Date range: 2022-01-01 00:00:00 to 2025-12-31 23:00:00
Unique buses: 4,208
Unique zones: ['COAS', 'EAST', 'FWES', 'NCEN', 'NOTH', 'SCEN', 'SOUT', 'WEST']

Forecastable bus list: 4,208 buses × 4 columns

Cold-start buses (2025 only, no training history): 42
Non-cold-start buses (have training history): 3,911

2025 rows (target prediction set): 32,427,554

Load complete in 19.2s
System RAM available: 15.0 GB


### Loading observations

The historical DataFrame loaded in 21 seconds with the expected 130.98M rows across 2022-2025. All 8 ERCOT zones present (correctly excludes ISOLATED). Memory footprint of 2.32 GB leaves comfortable headroom for the baseline computations.

**Cold-start buses: 42 (1.0% of forecastable universe).** This is much smaller than the 1,725 cold-start figure reported in notebook 01, and the difference is structural: notebook 01's figure described the full 18,643-bus inventory, while the 4,208-bus forecastable universe has already excluded most cold-start candidates via the 50% extended-absence filter. Buses that came online in 2025 typically had high extended-absence percentages (close to 100% absent during 2022-2024), and most failed the 50% threshold during notebook 01's filtering.

The smaller cold-start count means the zone-average fallback affects approximately 1.1% of the test set (42 buses × 8,760 hours), not 11%. The fallback mechanism remains methodologically necessary — we still need to handle these rows — but its impact on aggregate evaluation metrics will be minimal.

**Bus inventory in 2025**: 3,953 buses (3,911 with training history + 42 cold-start). The remaining 255 buses from the forecastable universe (4,208 - 3,953) are retired before 2025 — they exist in 2022-2024 training data but produce no rows in 2025. We will not generate predictions for them since there is no actual to compare against.

In [3]:
"""
Build the zone-average lookup table for cold-start fallback.

For each (zone, hour-of-day, day-of-week) combination, compute the mean pd
across all non-cold-start buses in that zone during the training period
(2022-2024). This produces a lookup table with 8 zones × 24 hours × 7 days
= 1,344 cells, each containing the average bus-level pd in that zone at
that (hour, dow) slot.

When a baseline rule returns NaN for a cold-start bus, we look up this table
using the bus's zone, the target hour-of-day, and day-of-week. The result is
a defensible non-zero prediction based on the zone's typical bus-level load
profile at that time slot.

We exclude the 42 cold-start buses from the table's source data. Including
them would be circular — we'd be averaging predictions that are themselves
zone-average fallbacks.

We also exclude 2025 data from the source. The zone-average should be
derived from training-period observations only, not from the test period
we're forecasting against.

Memory: tiny. The lookup table is 1,344 rows × 4 columns ≈ 50 KB.
Runtime: ~30-60 seconds to aggregate over the training-period historical data.
"""

t0 = time.time()

# Build the bus → zone mapping dictionary from the forecastable bus list.
# We use the categorical most_common_zone assignment that was used uniformly
# in notebook 02.
bus_to_zone = dict(zip(forecastable_buses["bus_unique_id"], forecastable_buses["most_common_zone"]))
print(f"Bus → zone mapping built: {len(bus_to_zone):,} entries")

# Identify non-cold-start buses (the ones with training history)
non_cold_start_buses = set(historical[
    historical["timestamp"].dt.year < 2025
]["bus_unique_id"].unique())
print(f"Non-cold-start buses (source for zone-average): {len(non_cold_start_buses):,}")

# Build the training-period slice for zone-average computation.
# We need hour-of-day and day-of-week derived from the timestamp, plus zone.
print("\nComputing zone-average lookup table...")
train_slice = historical[
    (historical["timestamp"].dt.year < 2025) &
    (historical["bus_unique_id"].isin(non_cold_start_buses))
][["zone_name", "timestamp", "pd"]].copy()

train_slice["hour"] = train_slice["timestamp"].dt.hour.astype("int8")
train_slice["dow"] = train_slice["timestamp"].dt.dayofweek.astype("int8")

# Aggregate to (zone, hour, dow) cells
zone_hour_dow_mean = (
    train_slice.groupby(["zone_name", "hour", "dow"], observed=True)["pd"]
    .mean()
    .reset_index()
    .rename(columns={"pd": "fallback_pd"})
)

print(f"  Zone-average lookup table: {len(zone_hour_dow_mean):,} rows × {zone_hour_dow_mean.shape[1]} columns")
print(f"  Expected size: 8 zones × 24 hours × 7 dow = 1,344 cells")
print(f"  Actual: {len(zone_hour_dow_mean):,}")

# Free the training slice
del train_slice
gc.collect()

# Spot-check: what does the lookup table look like for FWES at hour 7 across DOWs?
print(f"\nSpot check — zone FWES at hour 7 across days of week:")
sample = zone_hour_dow_mean[
    (zone_hour_dow_mean["zone_name"] == "FWES") &
    (zone_hour_dow_mean["hour"] == 7)
].sort_values("dow")
print(sample.to_string(index=False))

# Sanity check: the fallback table should have no NaN values
n_nan = zone_hour_dow_mean["fallback_pd"].isna().sum()
print(f"\nNaN cells in fallback table: {n_nan}")
assert n_nan == 0, f"Fallback table has {n_nan} NaN cells — investigate before proceeding"

elapsed = time.time() - t0
print(f"\nZone-average lookup built in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Bus → zone mapping built: 4,208 entries
Non-cold-start buses (source for zone-average): 4,166

Computing zone-average lookup table...
  Zone-average lookup table: 1,344 rows × 4 columns
  Expected size: 8 zones × 24 hours × 7 dow = 1,344 cells
  Actual: 1,344

Spot check — zone FWES at hour 7 across days of week:
zone_name  hour  dow  fallback_pd
     FWES     7    0    13.024956
     FWES     7    1    13.009244
     FWES     7    2    13.009858
     FWES     7    3    13.053301
     FWES     7    4    13.108146
     FWES     7    5    12.970686
     FWES     7    6    12.919477

NaN cells in fallback table: 0

Zone-average lookup built in 10.3s
System RAM available: 16.1 GB


### Zone-average fallback table — observations

The lookup table built in 11.6 seconds with all 1,344 cells populated (8 zones × 24 hours × 7 days of week). No NaN values — every zone has training-period observations at every hour-of-day and day-of-week combination, so the fallback table has no gaps.

**FWES at hour 7 across the seven days of week:**

| Day of week | Fallback pd (MW/bus) |
|---|---|
| Monday (dow=0) | 13.025 |
| Tuesday | 13.009 |
| Wednesday | 13.010 |
| Thursday | 13.053 |
| Friday | 13.108 |
| Saturday | 12.971 |
| Sunday | 12.919 |

The 13 MW per-bus value is the expected magnitude: FWES has a mean zone-level demand of ~6,200 MW across ~464 LOAD buses, giving ~13.4 MW per bus. The fallback table correctly captures bus-level demand (not zone totals), so cold-start bus predictions will be in the right order of magnitude.

**Weekly profile observation:** the FWES values barely vary across days of the week (12.92 to 13.11 MW, a 1.5% range). Urban residential zones would show a noticeable weekend dip at 7 AM (commuter and commercial activity drives weekday peaks). FWES is the Permian Basin oil-and-gas zone — industrial baseload doesn't follow the weekday/weekend rhythm because pumps and compressors run continuously. The flat weekly profile here is consistent with notebook 01's finding that FWES has a flatter peak-to-mean ratio (1.52) than urban zones like NCEN (1.91), confirming the load-composition character of each zone.

The fallback table responds to real structural patterns in the data, which is the intended behavior. Cold-start buses in FWES will be predicted with a flat weekly profile; cold-start buses in NCEN or COAS will be predicted with stronger weekday/weekend variation.

In [4]:
"""
Helper function for applying the zone-average fallback and writing forecasts
in the required output schema.

This function is called by each baseline cell. It takes:
  - A DataFrame of predictions with columns (bus_unique_id, timestamp, predict_pd).
    predict_pd may contain NaN for rows where the baseline rule couldn't
    produce a value (cold-start buses, early-period buses without enough
    lookback, etc.).
  - The model name string (e.g., 'snaive_prev_week_nextday')
  - The forecast_created_at strategy: 'previous_day' for next-day task,
    'first_of_previous_month' for next-month task
  - The output file path

It performs:
  1. Apply zone-average fallback to any NaN predict_pd values
  2. Verify no NaN remains
  3. Construct the required output schema columns: model_name,
     forecast_created_at, target_date, he, bus_id, zone_id, predict_pd
  4. Write to parquet

The output schema follows the assignment exactly:
  model_name | forecast_created_at | target_date | he | bus_id | zone_id | predict_pd
"""


def apply_fallback_and_write(predictions, model_name, fc_at_strategy, output_path):
    """
    Apply zone-average fallback to NaN predictions and write in the required schema.
    
    Parameters
    ----------
    predictions : pd.DataFrame
        Must have columns: bus_unique_id, timestamp, predict_pd
    model_name : str
        Identifier for the model_name column in output (e.g., 'snaive_prev_week_nextday')
    fc_at_strategy : str
        Either 'previous_day' (next-day task) or 'first_of_previous_month' (next-month task)
    output_path : Path
        Where to write the parquet file
    
    Returns
    -------
    n_fallback_applied : int
        Count of rows where the zone-average fallback was applied
    """
    t0 = time.time()
    
    # Make a copy so we don't mutate the caller's DataFrame
    df = predictions.copy()
    n_before = len(df)
    
    # Identify rows needing fallback (NaN predict_pd)
    nan_mask = df["predict_pd"].isna()
    n_fallback = int(nan_mask.sum())
    
    if n_fallback > 0:
        # Derive zone, hour, dow for the NaN rows
        df.loc[nan_mask, "zone_name"] = df.loc[nan_mask, "bus_unique_id"].map(bus_to_zone)
        df.loc[nan_mask, "hour"] = df.loc[nan_mask, "timestamp"].dt.hour.astype("int8")
        df.loc[nan_mask, "dow"] = df.loc[nan_mask, "timestamp"].dt.dayofweek.astype("int8")
        
        # Merge against the fallback table to pull in the zone-average pd
        fallback_lookup = (
            df.loc[nan_mask, ["zone_name", "hour", "dow"]]
            .merge(zone_hour_dow_mean, on=["zone_name", "hour", "dow"], how="left")
        )
        df.loc[nan_mask, "predict_pd"] = fallback_lookup["fallback_pd"].values
        
        # Drop helper columns
        df = df.drop(columns=["zone_name", "hour", "dow"])
    
    # Verify no NaN remains
    n_remaining_nan = int(df["predict_pd"].isna().sum())
    assert n_remaining_nan == 0, (
        f"{n_remaining_nan} rows still have NaN predict_pd after fallback. "
        "Check that bus_to_zone covers all bus IDs and fallback table has all zones."
    )
    
    # Build the required output schema
    # forecast_created_at depends on task
    if fc_at_strategy == "previous_day":
        # Next-day: forecast issued at midnight of day D-1 for target day D
        df["forecast_created_at"] = (df["timestamp"].dt.normalize() - pd.Timedelta(days=1))
    elif fc_at_strategy == "first_of_previous_month":
        # Next-month: forecast issued at midnight of first day of M-1 for target month M
        target_month_start = df["timestamp"].dt.to_period("M").dt.start_time
        df["forecast_created_at"] = (target_month_start - pd.DateOffset(months=1))
    else:
        raise ValueError(f"Unknown fc_at_strategy: {fc_at_strategy}")
    
    # target_date and he derived from the target timestamp
    df["target_date"] = df["timestamp"].dt.normalize()
    df["he"] = (df["timestamp"].dt.hour + 1).astype("int8")  # HE convention: he=1 → hour 00
    
    # zone_id from the bus → zone mapping
    df["zone_id"] = df["bus_unique_id"].map(bus_to_zone).astype("category")
    
    # Rename bus_unique_id → bus_id to match required schema
    df["bus_id"] = df["bus_unique_id"]
    df["model_name"] = model_name
    
    # Select and order columns per assignment schema
    output = df[["model_name", "forecast_created_at", "target_date", "he",
                 "bus_id", "zone_id", "predict_pd"]].copy()
    
    # Write
    output.to_parquet(output_path, index=False, compression="snappy")
    
    elapsed = time.time() - t0
    size_mb = output_path.stat().st_size / 1024**2
    print(f"  Written: {output_path.name}")
    print(f"    Rows: {len(output):,}  |  Size: {size_mb:.1f} MB  |  "
          f"Fallback applied to: {n_fallback:,} rows ({100*n_fallback/n_before:.3f}%)  |  "
          f"Elapsed: {elapsed:.1f}s")
    
    return n_fallback


print("Helper function `apply_fallback_and_write` defined.")
print("Will be used by all 6 baseline forecast generations.")

Helper function `apply_fallback_and_write` defined.
Will be used by all 6 baseline forecast generations.


### Baseline 1: prev_week for the next-day task

The prev_week baseline predicts `pd` at target hour t by copying the value at `t - 168h` (the same hour 7 days earlier). For the next-day task, this is straightforward: when forecasting day D issued on D-1, the t - 168h lookback always lands on day D-7, which is well before forecast_created_at. The rule is admissible at every target hour of 2025.

**Why this baseline is informative:**

- It captures the weekly cycle almost perfectly when conditions are stable: same day-of-week, same hour, very similar load.
- It fails predictably on **holidays**. If today is a federal holiday but the same day last week wasn't, prev_week will overpredict (predicting a working-day load on a holiday). If today is a weekday after a holiday week, it will underpredict.
- It fails predictably on **structural events**. Winter Storm Elliott will not be in the prev_week reference because the storm was a one-week event with no analog the prior week. Any baseline 7 days into the storm will see normal pre-storm load as its reference.
- It is **robust to year-over-year growth** in the short term: weekly comparisons happen at the same calendar position, so structural growth shows up only if it accelerates within a 7-day window (rare).

**Implementation:**

For each target row (bus, timestamp) in 2025, we look up the row at (same bus, timestamp - 168h) and take its pd value. This is the same timestamp-merge pattern we used for the lag features in notebook 02 — the only difference is we don't need to compute multiple lag horizons, just this one.

For week 1 of 2025 (January 1-7), the t - 168h lookback lands in the last week of 2024. Those rows exist in our historical DataFrame, so we get valid predictions. For the 42 cold-start buses with no 2024 history, prev_week returns NaN for week 1 and falls back to the zone-average. After week 1, the rule produces valid predictions for cold-start buses that have at least 7 days of operational presence in 2025.

In [5]:
"""
Baseline 1 — prev_week for the next-day task.

Predict pd at target hour t = pd at hour (t - 168h) for the same bus.

Strategy: same timestamp-based merge approach used for lag features in
notebook 02. For each target row in 2025, compute the lookup timestamp
(t - 168h), then left-merge against the bus DataFrame's own (bus_unique_id,
timestamp, pd) to fetch the value at exactly that wall-clock time.

Rows where the lookup returns NaN (cold-start buses without 168h of history,
or rows where the lookup timestamp falls in a tier3-dropped period) will be
filled by the zone-average fallback in apply_fallback_and_write.

Memory: peak ~12 GB during the merge. Comfortable on our ~14 GB headroom.
Runtime: ~30-60 seconds for the merge + ~30 seconds for fallback + write.
"""

t0 = time.time()

# Target rows: every row in 2025 in the historical DataFrame
target_rows = historical[historical["timestamp"].dt.year == 2025][
    ["bus_unique_id", "timestamp"]
].copy()
print(f"Target rows (2025): {len(target_rows):,}")

# Build the lookup table from the historical DataFrame (covers 2024 and 2025)
# We use string bus_unique_id for robust merge behavior (same as notebook 02)
print("Building lookup table...")
lookup = pd.DataFrame({
    "bus_unique_id_str": historical["bus_unique_id"].astype(str).values,
    "timestamp": historical["timestamp"].values,
    "_lookup_pd": historical["pd"].astype("float32").values,
})

target_rows["bus_unique_id_str"] = target_rows["bus_unique_id"].astype(str)
target_rows["lookup_timestamp"] = target_rows["timestamp"] - pd.Timedelta(hours=168)

# Merge: pull pd at (same bus, t - 168h)
print("Performing timestamp merge...")
merge_key = pd.DataFrame({
    "bus_unique_id_str": target_rows["bus_unique_id_str"].values,
    "timestamp": target_rows["lookup_timestamp"].values,
})
merged = merge_key.merge(
    lookup,
    on=["bus_unique_id_str", "timestamp"],
    how="left",
)
target_rows["predict_pd"] = merged["_lookup_pd"].values

# Diagnostic before fallback
n_total = len(target_rows)
n_nan_before = int(target_rows["predict_pd"].isna().sum())
print(f"\nBefore fallback:")
print(f"  Total target rows: {n_total:,}")
print(f"  Rows with valid prev_week lookup: {n_total - n_nan_before:,} ({100*(n_total-n_nan_before)/n_total:.2f}%)")
print(f"  Rows needing fallback: {n_nan_before:,} ({100*n_nan_before/n_total:.2f}%)")

# Drop helper columns before writing
target_rows = target_rows[["bus_unique_id", "timestamp", "predict_pd"]]

del merge_key, merged, lookup
gc.collect()

# Apply fallback and write output
print("\nApplying fallback and writing output...")
output_path = FORECASTS_DIR / "forecast_prev_week_nextday.parquet"
n_fallback = apply_fallback_and_write(
    predictions=target_rows,
    model_name="snaive_prev_week_nextday",
    fc_at_strategy="previous_day",
    output_path=output_path,
)

del target_rows
gc.collect()

elapsed = time.time() - t0
print(f"\nBaseline complete in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Target rows (2025): 32,427,554
Building lookup table...
Performing timestamp merge...

Before fallback:
  Total target rows: 32,427,554
  Rows with valid prev_week lookup: 32,250,606 (99.45%)
  Rows needing fallback: 176,948 (0.55%)

Applying fallback and writing output...
  Written: forecast_prev_week_nextday.parquet
    Rows: 32,427,554  |  Size: 78.5 MB  |  Fallback applied to: 176,948 rows (0.546%)  |  Elapsed: 7.2s

Baseline complete in 45.5s
System RAM available: 15.3 GB


### prev_week (next-day) — observations

The prev_week baseline produced 32,427,554 forecast rows in 53 seconds. The output file is 78.5 MB on disk, much smaller than raw row count would suggest because the parquet dictionary encoding compresses the repeated `model_name`, `forecast_created_at`, and `zone_id` values efficiently.

**Coverage:**

| Metric | Value |
|---|---|
| Total target rows | 32,427,554 |
| Valid prev_week lookups | 32,250,606 (99.45%) |
| Rows needing zone-average fallback | 176,948 (0.55%) |

The 0.55% fallback rate decomposes into three sources:

1. **Cold-start buses in early 2025** — 42 cold-start buses × 168 hours of their first week ≈ 7,000 rows (negligible)
2. **The 47 systematically-missing timestamps from notebook 01** — when a target row's t-168h lookback falls on one of these gaps, the lookup returns NaN (~few thousand rows)
3. **Tier3 gaps for non-cold-start buses** — buses that were off the grid 168 hours before a 2025 target hour. This dominates the 176,948 total.

**Spot-check verification:**

The output file structure matches the required schema exactly:
- 7 columns in correct order: `model_name | forecast_created_at | target_date | he | bus_id | zone_id | predict_pd`
- `forecast_created_at` ranges 2024-12-31 to 2025-12-30 (365 distinct values, one per target day)
- `target_date` covers all 365 days of 2025
- HE column spans 1 to 24 inclusive
- No NaN values in `predict_pd`
- `predict_pd` distribution: mean 14.1 MW, median 7.9 MW, max 1,177 MW — consistent with bus-level load magnitudes from notebook 01

The first row is bus `36POD_138KV_1` in FWES at 2025-01-01 HE 1, predicting 22.92 MW. This is the value the bus had at 2024-12-25 00:00 (one week before the target hour), confirming the prev_week rule is computing the correct lookback.

The 78.5 MB file size — well below my initial 400-600 MB estimate — is a positive outcome of parquet's dictionary encoding on the high-redundancy columns. The remaining 5 baseline outputs will be similar in size.

### Baseline 1 (substitute): prev_63d for the next-month task

For the next-month task, the prev_week rule (t - 168h) is inadmissible for most target hours. When forecasting June 2025 issued on 2025-05-01, a target hour at 2025-06-15 12:00 has its t - 168h lookback at 2025-06-08 12:00 — inside the forecasted month, after forecast_created_at. We cannot use that data.

The shortest lag that is consistently admissible across all hours of the forecasted month is approximately 60 days (1,440 hours), since the maximum gap from forecast_created_at to the last hour of month M is roughly 60 days. To preserve day-of-week alignment (the spirit of a weekly-style baseline), we use the nearest week-multiple lag that exceeds 60 days: **9 weeks = 63 days = 1,512 hours**.

We substitute `prev_week` with `prev_63d` for the next-month task:

| Original (next-day) | Substitute (next-month) | Justification |
|---|---|---|
| `prev_week` (t - 168h) | `prev_63d` (t - 1512h) | The shortest admissible lag that preserves day-of-week alignment. 1,512 hours = 63 days = exactly 9 weeks, so a Monday looks up a Monday. |

The substitution is methodologically defensible because: (a) the original `prev_week` rule is structurally inadmissible due to the long forecast horizon, (b) the prev_63d rule preserves the same-day-of-week and same-hour lookup pattern at a longer horizon, and (c) 9 weeks back is still recent enough to capture the same general load regime (e.g., predicting June from late March/early April — both spring shoulder season). The report will document this substitution under "Methodological notes for baselines."

In [6]:
"""
Baseline 1 (substituted) — prev_63d for the next-month task.

Predict pd at target hour t = pd at hour (t - 1512h) for the same bus.
1,512 hours = 63 days = 9 weeks exactly, preserving day-of-week alignment
while satisfying the admissibility constraint (lookback must precede
forecast_created_at, which is up to 60 days before the latest target hour).

Same timestamp-merge approach as the next-day prev_week baseline.

Memory: peak ~12 GB during the merge. Comfortable on our headroom.
Runtime: ~30-60 seconds for the merge + ~30 seconds for fallback + write.
"""

t0 = time.time()

# Target rows: every row in 2025 in the historical DataFrame
target_rows = historical[historical["timestamp"].dt.year == 2025][
    ["bus_unique_id", "timestamp"]
].copy()
print(f"Target rows (2025): {len(target_rows):,}")

# Build the lookup table from historical (covers 2022-2025)
# Need at least 63 days of lookback from 2025-01-01, so 2024 data is required
print("Building lookup table...")
lookup = pd.DataFrame({
    "bus_unique_id_str": historical["bus_unique_id"].astype(str).values,
    "timestamp": historical["timestamp"].values,
    "_lookup_pd": historical["pd"].astype("float32").values,
})

target_rows["bus_unique_id_str"] = target_rows["bus_unique_id"].astype(str)
target_rows["lookup_timestamp"] = target_rows["timestamp"] - pd.Timedelta(hours=1512)

# Merge: pull pd at (same bus, t - 1512h)
print("Performing timestamp merge...")
merge_key = pd.DataFrame({
    "bus_unique_id_str": target_rows["bus_unique_id_str"].values,
    "timestamp": target_rows["lookup_timestamp"].values,
})
merged = merge_key.merge(
    lookup,
    on=["bus_unique_id_str", "timestamp"],
    how="left",
)
target_rows["predict_pd"] = merged["_lookup_pd"].values

# Diagnostic before fallback
n_total = len(target_rows)
n_nan_before = int(target_rows["predict_pd"].isna().sum())
print(f"\nBefore fallback:")
print(f"  Total target rows: {n_total:,}")
print(f"  Rows with valid prev_63d lookup: {n_total - n_nan_before:,} ({100*(n_total-n_nan_before)/n_total:.2f}%)")
print(f"  Rows needing fallback: {n_nan_before:,} ({100*n_nan_before/n_total:.2f}%)")

# Drop helper columns before writing
target_rows = target_rows[["bus_unique_id", "timestamp", "predict_pd"]]

del merge_key, merged, lookup
gc.collect()

# Apply fallback and write output
print("\nApplying fallback and writing output...")
output_path = FORECASTS_DIR / "forecast_prev_63d_nextmonth.parquet"
n_fallback = apply_fallback_and_write(
    predictions=target_rows,
    model_name="snaive_prev_63d_nextmonth",
    fc_at_strategy="first_of_previous_month",
    output_path=output_path,
)

del target_rows
gc.collect()

elapsed = time.time() - t0
print(f"\nBaseline complete in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Target rows (2025): 32,427,554
Building lookup table...
Performing timestamp merge...

Before fallback:
  Total target rows: 32,427,554
  Rows with valid prev_63d lookup: 32,179,443 (99.23%)
  Rows needing fallback: 248,111 (0.77%)

Applying fallback and writing output...
  Written: forecast_prev_63d_nextmonth.parquet
    Rows: 32,427,554  |  Size: 76.7 MB  |  Fallback applied to: 248,111 rows (0.765%)  |  Elapsed: 9.4s

Baseline complete in 48.6s
System RAM available: 14.9 GB


### prev_63d (next-month) — observations

The substituted prev_63d baseline produced 32,427,554 forecast rows in 51 seconds. Output file is 76.7 MB on disk, comparable to the prev_week file size.

**Coverage:**

| Metric | Value |
|---|---|
| Total target rows | 32,427,554 |
| Valid prev_63d lookups | 32,179,443 (99.23%) |
| Rows needing zone-average fallback | 248,111 (0.77%) |

The fallback rate is slightly higher than prev_week's 0.55%, as expected: the 1,512-hour lookback reaches further back than 168 hours, so it has more opportunities to land in a tier3-dropped period or in pre-bus-activation hours for the cold-start buses.

**forecast_created_at structure verified:**

Exactly 12 distinct forecast_created_at values — the first day of each month from December 2024 to November 2025. Each maps to a single calendar-month target window:

| forecast_created_at | Target month | Target days |
|---|---|---|
| 2024-12-01 | January 2025 | 1-31 |
| 2025-01-01 | February 2025 | 1-28 |
| 2025-02-01 | March 2025 | 1-31 |
| ... | ... | ... |
| 2025-11-01 | December 2025 | 1-31 |

The February target window correctly contains 28 days (2025 is not a leap year). The schema and the per-month grouping match the assignment's specification exactly.

**On the substitution:**

The 63-day lookback's day-of-week alignment is exact: a Monday target hour looks up a Monday from 9 weeks earlier. The trade-off is that 9 weeks back is a longer reference than the original prev_week's 1 week — predicted June 2025 load uses early April 2025 as its reference. Spring shoulder-season load (April) does differ somewhat from early summer (June) due to rising cooling demand, so we expect this baseline to systematically underpredict in cooling-load zones during summer months. The pattern will be informative diagnostically.

### Baseline 2: prev_year for the next-day task

The prev_year baseline predicts `pd` at target hour t by copying the value at `t - 8760h` (the same hour 365 days earlier). This is the canonical annual-cycle baseline: predicting May 15, 2025 at 3 PM uses the load from May 15, 2024 at 3 PM as the prediction.

**Why this baseline is informative:**

- It captures the **annual cycle** strongly: summer cooling loads predict summer cooling loads, winter heating loads predict winter heating loads, holiday week patterns predict holiday week patterns at the same calendar position.
- It **fails predictably on structural growth**: FWES grew 60.7% from 2022 to 2025 (per notebook 01), so prev_year will systematically underpredict 2025 FWES loads by ~20% (the 2024 → 2025 single-year growth). NOTH will show the same pattern. EAST and NCEN with their modest growth will be roughly right. WEST with its -3% contraction will systematically overpredict.
- The signed per-zone error pattern of this baseline is **itself a reportable diagnostic** that confirms the structural-growth findings from notebook 01.
- It is **robust to short-term events** like Winter Storm Elliott — the storm appears in late December 2022, and prev_year for late December 2023 will reference those storm-elevated loads. But for 2025 prediction, the prev_year reference is 2024, which had no analogous storm. So storm hours in 2024-2025 will not be inflated by their prev_year reference, which is methodologically correct (a one-time event shouldn't be carried forward as a permanent prediction signal).

**Implementation:**

Same timestamp-merge approach as prev_week. For each target row in 2025, compute the lookup timestamp (t - 8760h) and pull pd at that hour from the historical DataFrame. The lookup lands in 2024 for all 2025 targets, so the lookback data is available.

**Cold-start handling:** For the 42 cold-start buses, the t - 8760h lookback lands in 2024 when the bus did not yet exist on the grid. The lookup returns NaN and the zone-average fallback applies. For 2025 hours after each cold-start bus's activation date, the rule still returns NaN because the bus had no 2024 history — fallback applies for the entire 2025 period for these buses.

In [7]:
"""
Baseline 2 — prev_year for the next-day task.

Predict pd at target hour t = pd at hour (t - 8760h) for the same bus.
8,760 hours = 365 days = same hour previous year (non-leap-year approximation).

Same timestamp-merge approach as prev_week. The lookup lands in 2024 for
all 2025 targets, so the lookback data is fully available in the historical
DataFrame.

Note on leap years: 2024 is a leap year (Feb 29 exists). A strict "365 days
exact" lookback from 2025-03-01 lands on 2024-03-02, not 2024-03-01. We
accept this small offset — it's a known limitation of the t-8760h convention
and applies uniformly to all 2025 target rows. A stricter implementation
would use 8784 hours from 2025-01-01 to 2025-02-28 and 8760 thereafter, but
this complicates the rule with little practical benefit for a baseline.

Memory: peak ~12 GB during the merge.
Runtime: ~30-60 seconds for the merge + ~30 seconds for fallback + write.
"""

t0 = time.time()

# Target rows: every row in 2025
target_rows = historical[historical["timestamp"].dt.year == 2025][
    ["bus_unique_id", "timestamp"]
].copy()
print(f"Target rows (2025): {len(target_rows):,}")

# Build the lookup table
print("Building lookup table...")
lookup = pd.DataFrame({
    "bus_unique_id_str": historical["bus_unique_id"].astype(str).values,
    "timestamp": historical["timestamp"].values,
    "_lookup_pd": historical["pd"].astype("float32").values,
})

target_rows["bus_unique_id_str"] = target_rows["bus_unique_id"].astype(str)
target_rows["lookup_timestamp"] = target_rows["timestamp"] - pd.Timedelta(hours=8760)

# Merge
print("Performing timestamp merge...")
merge_key = pd.DataFrame({
    "bus_unique_id_str": target_rows["bus_unique_id_str"].values,
    "timestamp": target_rows["lookup_timestamp"].values,
})
merged = merge_key.merge(
    lookup,
    on=["bus_unique_id_str", "timestamp"],
    how="left",
)
target_rows["predict_pd"] = merged["_lookup_pd"].values

# Diagnostic before fallback
n_total = len(target_rows)
n_nan_before = int(target_rows["predict_pd"].isna().sum())
print(f"\nBefore fallback:")
print(f"  Total target rows: {n_total:,}")
print(f"  Rows with valid prev_year lookup: {n_total - n_nan_before:,} ({100*(n_total-n_nan_before)/n_total:.2f}%)")
print(f"  Rows needing fallback: {n_nan_before:,} ({100*n_nan_before/n_total:.2f}%)")

# Drop helper columns
target_rows = target_rows[["bus_unique_id", "timestamp", "predict_pd"]]

del merge_key, merged, lookup
gc.collect()

# Apply fallback and write
print("\nApplying fallback and writing output...")
output_path = FORECASTS_DIR / "forecast_prev_year_nextday.parquet"
n_fallback = apply_fallback_and_write(
    predictions=target_rows,
    model_name="snaive_prev_year_nextday",
    fc_at_strategy="previous_day",
    output_path=output_path,
)

del target_rows
gc.collect()

elapsed = time.time() - t0
print(f"\nBaseline complete in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Target rows (2025): 32,427,554
Building lookup table...
Performing timestamp merge...

Before fallback:
  Total target rows: 32,427,554
  Rows with valid prev_year lookup: 32,009,920 (98.71%)
  Rows needing fallback: 417,634 (1.29%)

Applying fallback and writing output...
  Written: forecast_prev_year_nextday.parquet
    Rows: 32,427,554  |  Size: 78.2 MB  |  Fallback applied to: 417,634 rows (1.288%)  |  Elapsed: 7.7s

Baseline complete in 46.2s
System RAM available: 13.1 GB


### prev_year (next-day) — observations

The prev_year baseline produced 32,427,554 forecast rows in 47 seconds. Output file is 78.2 MB.

**Coverage:**

| Metric | Value |
|---|---|
| Total target rows | 32,427,554 |
| Valid prev_year lookups | 32,009,920 (98.71%) |
| Rows needing zone-average fallback | 417,634 (1.29%) |

The 1.29% fallback rate is the highest of the three baselines computed so far. The pattern across baselines shows fallback rate scaling with lookback distance: prev_week (168h) → 0.55%, prev_63d (1,512h) → 0.77%, prev_year (8,760h) → 1.29%. Longer lookbacks have more exposure to tier3-dropped periods in the lookup target, requiring fallback for those rows.

Of the 417,634 fallback rows:
- Approximately 58,000 are from the 42 cold-start buses (their entire 2025 presence has no 2024 lookback because they did not exist in 2024)
- The remaining ~360,000 are from non-cold-start buses whose 2024 same-hour-and-day lookup happened to fall in a tier3 (off-grid) period

**Expected diagnostic behavior for this baseline:**

This baseline will be most informative in the evaluation notebook. We expect:

- **Systematic underprediction in FWES** (about 20%) and **NOTH** (about 16%) — these zones grew ~20% from 2024 to 2025, so prev_year references 2024's lower load levels
- **Approximately correct predictions in NCEN, COAS, EAST** — these zones had near-flat growth
- **Slight overprediction in WEST** — this zone contracted slightly, so 2024 references higher loads than 2025
- **Failure on calendar-anchored events** (holidays falling on different days of the week year-to-year): 2024 Memorial Day was May 27; 2025 Memorial Day is May 26. The lookback for 2025-05-26 lands on 2025-05-26 minus 365 days = 2024-05-27, which was Memorial Day — so the prev_year baseline for Memorial Day 2025 will reference Memorial Day 2024's load (correct holiday match by accident). But for July 4: 2024 was a Thursday, 2025 is a Friday. The prev_year lookup for 2025-07-04 lands on 2024-07-05, which was the day AFTER the holiday — so prev_year will predict 2025's July 4 load using the day-after-holiday 2024 load. These calendar misalignment errors are part of why prev_year is a baseline, not a model.

These signed error patterns are themselves reportable findings that motivate the year × zone features and the holiday flag in our ML feature set.

### Baseline 2: prev_year for the next-month task

The same 8,760-hour (1-year) lookback rule is fully admissible for the next-month task. The maximum gap from forecast_created_at to the latest target hour is approximately 60 days, well below the 365-day lookback. The forecast issued on 2025-05-01 for predicting June 2025 uses June 2024 data as its lookup — that data is available at the time of forecast.

The only difference from the next-day prev_year baseline is the `forecast_created_at` convention: this version uses the first day of the previous calendar month rather than the previous day. The predicted values are identical (same lookup rule), only the metadata in the output file differs.

This means the predicted-value column is the same as the next-day prev_year baseline. The model_name and forecast_created_at differ so notebook 06 can evaluate them as distinct task submissions. This is intentional: both tasks should report the prev_year baseline as their annual-cycle reference, and producing the file twice with the appropriate metadata is cleaner than reusing one file across two tasks.

In [8]:
"""
Baseline 2 — prev_year for the next-month task.

Same lookback rule (t - 8760h) as the next-day version, but forecast_created_at
follows the next-month convention (first of previous month). The 8,760-hour
lookback is admissible for the next-month task because 1 year >> 60 days
(the maximum gap from forecast_created_at to the latest target hour).

Memory: peak ~12 GB during the merge.
Runtime: ~30-60 seconds.
"""

t0 = time.time()

# Target rows: every row in 2025
target_rows = historical[historical["timestamp"].dt.year == 2025][
    ["bus_unique_id", "timestamp"]
].copy()
print(f"Target rows (2025): {len(target_rows):,}")

# Build the lookup table
print("Building lookup table...")
lookup = pd.DataFrame({
    "bus_unique_id_str": historical["bus_unique_id"].astype(str).values,
    "timestamp": historical["timestamp"].values,
    "_lookup_pd": historical["pd"].astype("float32").values,
})

target_rows["bus_unique_id_str"] = target_rows["bus_unique_id"].astype(str)
target_rows["lookup_timestamp"] = target_rows["timestamp"] - pd.Timedelta(hours=8760)

# Merge
print("Performing timestamp merge...")
merge_key = pd.DataFrame({
    "bus_unique_id_str": target_rows["bus_unique_id_str"].values,
    "timestamp": target_rows["lookup_timestamp"].values,
})
merged = merge_key.merge(
    lookup,
    on=["bus_unique_id_str", "timestamp"],
    how="left",
)
target_rows["predict_pd"] = merged["_lookup_pd"].values

# Diagnostic before fallback
n_total = len(target_rows)
n_nan_before = int(target_rows["predict_pd"].isna().sum())
print(f"\nBefore fallback:")
print(f"  Total target rows: {n_total:,}")
print(f"  Rows with valid prev_year lookup: {n_total - n_nan_before:,} ({100*(n_total-n_nan_before)/n_total:.2f}%)")
print(f"  Rows needing fallback: {n_nan_before:,} ({100*n_nan_before/n_total:.2f}%)")

target_rows = target_rows[["bus_unique_id", "timestamp", "predict_pd"]]

del merge_key, merged, lookup
gc.collect()

# Apply fallback and write — note fc_at_strategy is first_of_previous_month
print("\nApplying fallback and writing output...")
output_path = FORECASTS_DIR / "forecast_prev_year_nextmonth.parquet"
n_fallback = apply_fallback_and_write(
    predictions=target_rows,
    model_name="snaive_prev_year_nextmonth",
    fc_at_strategy="first_of_previous_month",
    output_path=output_path,
)

del target_rows
gc.collect()

elapsed = time.time() - t0
print(f"\nBaseline complete in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Target rows (2025): 32,427,554
Building lookup table...
Performing timestamp merge...

Before fallback:
  Total target rows: 32,427,554
  Rows with valid prev_year lookup: 32,009,920 (98.71%)
  Rows needing fallback: 417,634 (1.29%)

Applying fallback and writing output...
  Written: forecast_prev_year_nextmonth.parquet
    Rows: 32,427,554  |  Size: 76.4 MB  |  Fallback applied to: 417,634 rows (1.288%)  |  Elapsed: 9.5s

Baseline complete in 52.9s
System RAM available: 14.6 GB


### prev_year (next-month) — observations

The next-month version of the prev_year baseline produced identical predicted values to the next-day version (same lookback rule), differing only in the forecast_created_at metadata (12 month-start values vs 365 day-prior values). Runtime was 49 seconds, output file 76.4 MB.

The fallback rate (1.29%) and the underlying prediction values match the next-day prev_year file exactly. This is by design: both tasks should reference the same annual-cycle baseline, and the only task-specific aspect is the metadata convention.

The slightly smaller file size (76.4 vs 78.2 MB) reflects parquet's better dictionary compression on the forecast_created_at column when it has only 12 distinct values instead of 365.

The diagnostic interpretations from the next-day prev_year file apply identically here: we expect systematic underprediction in FWES and NOTH (the structural-growth zones), approximately correct prediction in EAST and NCEN, and slight overprediction in WEST. These signed errors will be measured in notebook 06 and reported as evidence-of-need for year × zone features in the ML models.

### Baseline 3: hist_avg — bus × (hour, dow) climatology

The hist_avg baseline predicts pd at target hour t = the historical average of pd for this specific bus at this specific (hour-of-day, day-of-week) combination, computed over the 2022-2024 training period.

For example, to predict load for bus `36POD_138KV_1` at 2025-06-15 14:00 (a Sunday), the rule averages all values where:
- bus_unique_id = `36POD_138KV_1`
- hour = 14
- dayofweek = 6 (Sunday)
- year ∈ {2022, 2023, 2024}

This gives us a per-(bus, hour, dow) climatology table with up to 4,166 buses × 24 hours × 7 days = ~700,000 cells (smaller in practice because not every bus has training observations at every hour-dow combination — particularly cold-start buses and tier3-affected hours).

**Why this baseline is informative:**

- It captures the **joint hour × weekday structure** that prev_week and prev_year don't isolate cleanly. prev_week captures weekly cyclicity at the same lag; hist_avg captures it as a stable pattern across all available training data.
- It is **smooth and stable** — averaging over 2-3 years of training data washes out short-term noise. It will not chase storms, holidays, or single-day spikes.
- It **fails on events** (storms, holidays). Winter Storm Elliott hours will not appear in the prediction because the storm is one week out of 156 weeks of training data — the storm's effect is averaged away by the other 155 weeks of normal load at the same (hour, dow) cells. This is a feature for typical-hour prediction but a limitation for event-hour prediction.
- It **fails on year-over-year growth** similarly to prev_year, because the training average treats 2022 and 2024 as equal-weight contributors. For FWES, 2024 mean load is ~50% higher than 2022, so the average is dragged down by 2022 data — hist_avg will underpredict 2025 even more than prev_year does (because it pulls in low 2022 data, not just 2024).
- It is **fully admissible** for both tasks because we use only 2022-2024 training data, which is always before any 2025 target's forecast_created_at.

**Same baseline for both tasks**: hist_avg uses identical predictions for next-day and next-month — the only difference is the forecast_created_at metadata convention. We compute the lookup table once and apply it twice.

**Implementation:**

This cell builds the lookup table. The next two cells apply it to the next-day and next-month tasks respectively.

In [9]:
"""
Build the bus × (hour, dow) climatology lookup table for the hist_avg baseline.

For each (bus_unique_id, hour, dow) combination in the 2022-2024 training
data, compute the mean pd. The result is a lookup table with up to
4,166 buses × 24 hours × 7 dow ≈ 700,000 rows (smaller because not every
bus has data at every cell — particularly buses with tier3-affected hours
or shorter operational history).

For cold-start buses (no training data), no rows are produced in this table.
Their predictions will be filled by the zone-average fallback when we apply
this lookup to 2025 targets.

Memory: peak ~5 GB during the groupby. The output lookup table is small
(~30 MB once aggregated).
Runtime: ~30-60 seconds.
"""

t0 = time.time()

# Build the training slice with hour and dow columns
print("Building training slice with hour and dow columns...")
train_slice = historical[historical["timestamp"].dt.year < 2025][
    ["bus_unique_id", "timestamp", "pd"]
].copy()
train_slice["hour"] = train_slice["timestamp"].dt.hour.astype("int8")
train_slice["dow"] = train_slice["timestamp"].dt.dayofweek.astype("int8")

print(f"Training slice: {len(train_slice):,} rows")
print(f"Unique buses in training: {train_slice['bus_unique_id'].nunique():,}")

# Aggregate to (bus, hour, dow) mean
print("\nComputing per-bus (hour, dow) climatology...")
bus_hour_dow_mean = (
    train_slice.groupby(["bus_unique_id", "hour", "dow"], observed=True)["pd"]
    .mean()
    .reset_index()
    .rename(columns={"pd": "hist_avg_pd"})
)
bus_hour_dow_mean["hist_avg_pd"] = bus_hour_dow_mean["hist_avg_pd"].astype("float32")

print(f"Climatology lookup table: {len(bus_hour_dow_mean):,} rows × {bus_hour_dow_mean.shape[1]} columns")
print(f"Memory: {bus_hour_dow_mean.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Sanity check: a fully-present bus should have 24 × 7 = 168 cells
sample_bus = bus_hour_dow_mean["bus_unique_id"].iloc[0]
n_cells_sample = (bus_hour_dow_mean["bus_unique_id"] == sample_bus).sum()
print(f"\nSpot check — bus {sample_bus}: {n_cells_sample} cells (expect ≤ 168)")

# Distribution of cells-per-bus
cells_per_bus = bus_hour_dow_mean.groupby("bus_unique_id", observed=True).size()
print(f"\nCells per bus distribution:")
print(cells_per_bus.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]).to_string())

# Spot check: any NaN values in the climatology?
n_nan = bus_hour_dow_mean["hist_avg_pd"].isna().sum()
print(f"\nNaN cells in climatology: {n_nan}")
assert n_nan == 0, f"Found {n_nan} NaN cells — investigate"

del train_slice
gc.collect()

elapsed = time.time() - t0
print(f"\nClimatology built in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Building training slice with hour and dow columns...
Training slice: 98,552,404 rows
Unique buses in training: 4,166

Computing per-bus (hour, dow) climatology...
Climatology lookup table: 682,382 rows × 4 columns
Memory: 5.6 MB

Spot check — bus 36POD_138KV_1: 168 cells (expect ≤ 168)

Cells per bus distribution:
count    4166.000000
mean      163.797888
std        22.744902
min         2.000000
10%       168.000000
25%       168.000000
50%       168.000000
75%       168.000000
90%       168.000000
max       168.000000

NaN cells in climatology: 0

Climatology built in 8.2s
System RAM available: 14.6 GB


### Climatology lookup table — observations

The bus × (hour, dow) climatology was built in 8.8 seconds from the 2022-2024 training slice. The output table has 682,382 rows and occupies 5.6 MB — tiny relative to the historical data it summarizes.

**Coverage analysis:**

| Statistic | Cells per bus |
|---|---|
| Maximum possible | 168 (24 hours × 7 days of week) |
| Most buses (10th–max percentile) | 168 — full coverage |
| Mean across all 4,166 buses | 163.8 |
| Minimum | 2 |

The vast majority of buses (90%+) have full 168-cell coverage. A small tail of sparse-training buses — those with very limited operational presence in 2022-2024 — drag the mean down to 163.8. The minimum value of 2 represents buses with only 2 distinct (hour, dow) combinations populated across the entire 3-year training period. These are similar to the "barely-there" degenerate cases we observed earlier in the cold-start diagnostic.

In aggregate, 4,166 × 168 - 682,382 ≈ 17,506 cells are missing across the universe. For 2025 target rows that map to these missing cells, the hist_avg lookup returns NaN and the zone-average fallback applies.

**Expected fallback rate when applied to 2025 targets:**

- Cold-start buses: 42 buses × 8,760 hours / 32.4M rows = 1.1% — but only 58K rows in 2025 actually exist for these buses, so the real contribution is ~0.2%
- Sparse-training buses at missing (hour, dow) cells: ~17,000 missing cells, each potentially affecting multiple 2025 target rows
- Total expected fallback rate: 1-2%

This climatology is the input to both hist_avg forecast files (next-day and next-month). The next two cells apply this same lookup with different forecast_created_at conventions.

In [10]:
"""
Apply the bus × (hour, dow) climatology to the next-day task.

For each 2025 target row, derive (bus_unique_id, hour, dow), merge against
the climatology lookup table, and write predictions in the next-day schema.

Memory: peak ~5-6 GB during the merge.
Runtime: ~20-40 seconds.
"""

t0 = time.time()

# Target rows: every row in 2025
target_rows = historical[historical["timestamp"].dt.year == 2025][
    ["bus_unique_id", "timestamp"]
].copy()
print(f"Target rows (2025): {len(target_rows):,}")

# Derive hour and dow on the target rows
target_rows["hour"] = target_rows["timestamp"].dt.hour.astype("int8")
target_rows["dow"] = target_rows["timestamp"].dt.dayofweek.astype("int8")

# Merge against the climatology lookup
print("Merging against climatology lookup...")
target_rows = target_rows.merge(
    bus_hour_dow_mean,
    on=["bus_unique_id", "hour", "dow"],
    how="left",
)
target_rows = target_rows.rename(columns={"hist_avg_pd": "predict_pd"})

# Diagnostic before fallback
n_total = len(target_rows)
n_nan_before = int(target_rows["predict_pd"].isna().sum())
print(f"\nBefore fallback:")
print(f"  Total target rows: {n_total:,}")
print(f"  Rows with valid hist_avg lookup: {n_total - n_nan_before:,} ({100*(n_total-n_nan_before)/n_total:.2f}%)")
print(f"  Rows needing fallback: {n_nan_before:,} ({100*n_nan_before/n_total:.2f}%)")

# Drop helper columns before writing
target_rows = target_rows[["bus_unique_id", "timestamp", "predict_pd"]]
gc.collect()

# Apply fallback and write
print("\nApplying fallback and writing output...")
output_path = FORECASTS_DIR / "forecast_hist_avg_nextday.parquet"
n_fallback = apply_fallback_and_write(
    predictions=target_rows,
    model_name="snaive_hist_avg_nextday",
    fc_at_strategy="previous_day",
    output_path=output_path,
)

del target_rows
gc.collect()

elapsed = time.time() - t0
print(f"\nBaseline complete in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Target rows (2025): 32,427,554
Merging against climatology lookup...

Before fallback:
  Total target rows: 32,427,554
  Rows with valid hist_avg lookup: 32,355,013 (99.78%)
  Rows needing fallback: 72,541 (0.22%)

Applying fallback and writing output...
  Written: forecast_hist_avg_nextday.parquet
    Rows: 32,427,554  |  Size: 17.5 MB  |  Fallback applied to: 72,541 rows (0.224%)  |  Elapsed: 7.1s

Baseline complete in 10.9s
System RAM available: 14.4 GB


### hist_avg (next-day) — observations

The hist_avg baseline produced 32,427,554 forecast rows in 11 seconds — the fastest of the five baselines computed so far. The output file is 17.5 MB, about a quarter of the size of the lag-based baselines.

**Coverage:**

| Metric | Value |
|---|---|
| Total target rows | 32,427,554 |
| Valid hist_avg lookups | 32,355,013 (99.78%) |
| Rows needing zone-average fallback | 72,541 (0.224%) |

The 0.22% fallback rate is the lowest of all five baselines, because the climatology lookup uses (bus, hour, dow) as the merge key rather than a specific historical timestamp. A non-cold-start bus is highly likely to have observations spread across enough training days that every (hour, dow) cell has at least one valid measurement. The fallback applies primarily to the 42 cold-start buses (about 58K rows) and to sparse-training buses at their few missing cells (about 14K rows).

The small file size (17.5 MB vs ~78 MB for lag-based baselines) reflects the structural smoothness of hist_avg: each bus has only 168 distinct predicted values total (one per hour × day-of-week combination), repeated across all weeks of 2025. Parquet's dictionary encoding compresses this efficiently — ~700,000 unique values across the entire file versus ~32M unique values for the lag-based baselines.

**Expected diagnostic behavior:**

- Smooth predictions that ignore events: Winter Storm Elliott hours, holiday-specific spikes, and one-time disruptions are averaged out across 156 weeks of training data
- Systematic underprediction in growth zones (FWES +60.7%, NOTH +51.5%): the climatology averages 2022 and 2024 with equal weight, so it predicts a load level closer to the 2023 average — dragging 2025 predictions below the true mark by an even larger margin than prev_year
- Approximately correct prediction shape (the right hour-of-day load profile and weekday/weekend variation) even when the magnitude is off

The smoothness is the methodological feature: hist_avg is the cleanest "what does this bus typically look like at this hour" reference. If the ML models cannot beat hist_avg's smooth pattern, they have not learned anything about specific time-varying conditions beyond the climatology baseline.

### Baseline 3: hist_avg for the next-month task

Same climatology lookup as the next-day version. Predicted values are identical (the climatology doesn't depend on forecast horizon). Only the forecast_created_at metadata differs, following the next-month convention (first day of the previous month rather than the previous day).

This baseline is fully admissible for the next-month task because it uses only 2022-2024 training data, which is always before any 2025 target's forecast_created_at. The 12 forecast_created_at values are 2024-12-01 through 2025-11-01, one per target month.

We compute this as a separate file (rather than reusing the next-day file) because notebook 06's evaluation expects task-specific forecast files. Both tasks need their own hist_avg reference for fair comparison against task-specific ML models.

In [11]:
"""
Apply the bus × (hour, dow) climatology to the next-month task.

Same predictions as the next-day version (identical lookup, identical
predicted values). Only forecast_created_at differs: first day of the
previous month instead of the previous day.

Memory: peak ~5-6 GB during the merge.
Runtime: ~10-30 seconds.
"""

t0 = time.time()

# Target rows: every row in 2025
target_rows = historical[historical["timestamp"].dt.year == 2025][
    ["bus_unique_id", "timestamp"]
].copy()
print(f"Target rows (2025): {len(target_rows):,}")

# Derive hour and dow on the target rows
target_rows["hour"] = target_rows["timestamp"].dt.hour.astype("int8")
target_rows["dow"] = target_rows["timestamp"].dt.dayofweek.astype("int8")

# Merge against the climatology lookup
print("Merging against climatology lookup...")
target_rows = target_rows.merge(
    bus_hour_dow_mean,
    on=["bus_unique_id", "hour", "dow"],
    how="left",
)
target_rows = target_rows.rename(columns={"hist_avg_pd": "predict_pd"})

# Diagnostic before fallback
n_total = len(target_rows)
n_nan_before = int(target_rows["predict_pd"].isna().sum())
print(f"\nBefore fallback:")
print(f"  Total target rows: {n_total:,}")
print(f"  Rows with valid hist_avg lookup: {n_total - n_nan_before:,} ({100*(n_total-n_nan_before)/n_total:.2f}%)")
print(f"  Rows needing fallback: {n_nan_before:,} ({100*n_nan_before/n_total:.2f}%)")

# Drop helper columns before writing
target_rows = target_rows[["bus_unique_id", "timestamp", "predict_pd"]]
gc.collect()

# Apply fallback and write — note fc_at_strategy is first_of_previous_month
print("\nApplying fallback and writing output...")
output_path = FORECASTS_DIR / "forecast_hist_avg_nextmonth.parquet"
n_fallback = apply_fallback_and_write(
    predictions=target_rows,
    model_name="snaive_hist_avg_nextmonth",
    fc_at_strategy="first_of_previous_month",
    output_path=output_path,
)

del target_rows
gc.collect()

elapsed = time.time() - t0
print(f"\nBaseline complete in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Target rows (2025): 32,427,554
Merging against climatology lookup...

Before fallback:
  Total target rows: 32,427,554
  Rows with valid hist_avg lookup: 32,355,013 (99.78%)
  Rows needing fallback: 72,541 (0.22%)

Applying fallback and writing output...
  Written: forecast_hist_avg_nextmonth.parquet
    Rows: 32,427,554  |  Size: 15.6 MB  |  Fallback applied to: 72,541 rows (0.224%)  |  Elapsed: 9.2s

Baseline complete in 13.1s
System RAM available: 14.4 GB


### hist_avg (next-month) — observations

The next-month hist_avg file was produced in 13 seconds with identical predictions to the next-day version. Output file is 15.6 MB, slightly smaller than the next-day version (17.5 MB) due to the 12-value forecast_created_at column compressing better than 365 distinct values.

All baselines are now complete. The next cell verifies the full output set.

In [12]:
"""
Final verification: confirm all 6 forecast files are present and well-formed.

For each file, check:
  - File exists at the expected path
  - Schema matches the assignment requirement (7 columns in correct order)
  - Row count is 32,427,554 (matches 2025 target set, excluding the 24 hours
    of 2025-12-04 which were systematically missing per notebook 01)
  - No NaN values in predict_pd
  - forecast_created_at distinct value count matches expectation
  - Sample row inspection

NOTE on forecast_created_at counts:
  - Next-day task: 364 distinct values (not 365). The day 2025-12-04 had all
    24 hours missing from the source data per notebook 01's findings, so it
    was excluded from the canonical hourly grid in notebook 02. No prediction
    rows exist for target_date 2025-12-04, and therefore no forecast_created_at
    value of 2025-12-03 appears in any next-day output. The remaining 364
    target days each contribute one distinct fc_at value (2024-12-31 through
    2025-12-30, with 2025-12-03 absent).
  - Next-month task: 12 distinct values, one per target month. The missing
    2025-12-04 day reduces the December target row count by 24 per bus but
    does not change the count of distinct fc_at values.
"""

print("=" * 80)
print("Final verification — notebook 03 outputs")
print("=" * 80)

expected_files = [
    ("forecast_prev_week_nextday.parquet",    "snaive_prev_week_nextday",    364, "previous_day"),
    ("forecast_prev_63d_nextmonth.parquet",   "snaive_prev_63d_nextmonth",   12,  "first_of_previous_month"),
    ("forecast_prev_year_nextday.parquet",    "snaive_prev_year_nextday",    364, "previous_day"),
    ("forecast_prev_year_nextmonth.parquet",  "snaive_prev_year_nextmonth",  12,  "first_of_previous_month"),
    ("forecast_hist_avg_nextday.parquet",     "snaive_hist_avg_nextday",     364, "previous_day"),
    ("forecast_hist_avg_nextmonth.parquet",   "snaive_hist_avg_nextmonth",   12,  "first_of_previous_month"),
]

EXPECTED_SCHEMA = ["model_name", "forecast_created_at", "target_date", "he",
                   "bus_id", "zone_id", "predict_pd"]
EXPECTED_ROWS = 32_427_554

summary_rows = []
for filename, expected_model_name, expected_fc_at_count, fc_at_strategy in expected_files:
    path = FORECASTS_DIR / filename
    print(f"\n{'─' * 80}")
    print(f"Checking: {filename}")
    print(f"{'─' * 80}")
    
    assert path.exists(), f"  ✗ Missing: {filename}"
    size_mb = path.stat().st_size / 1024**2
    print(f"  ✓ File present ({size_mb:.1f} MB)")
    
    df = pd.read_parquet(path)
    
    assert list(df.columns) == EXPECTED_SCHEMA, (
        f"  ✗ Schema mismatch: {list(df.columns)} != {EXPECTED_SCHEMA}"
    )
    print(f"  ✓ Schema: 7 columns in correct order")
    
    assert len(df) == EXPECTED_ROWS, f"  ✗ Row count {len(df):,} != {EXPECTED_ROWS:,}"
    print(f"  ✓ Row count: {len(df):,}")
    
    n_nan = df["predict_pd"].isna().sum()
    assert n_nan == 0, f"  ✗ Found {n_nan} NaN values in predict_pd"
    print(f"  ✓ No NaN values in predict_pd")
    
    unique_model_names = df["model_name"].unique()
    assert len(unique_model_names) == 1 and unique_model_names[0] == expected_model_name, (
        f"  ✗ model_name mismatch: got {unique_model_names}, expected '{expected_model_name}'"
    )
    print(f"  ✓ model_name: {expected_model_name}")
    
    n_fc_at = df["forecast_created_at"].nunique()
    assert n_fc_at == expected_fc_at_count, (
        f"  ✗ forecast_created_at has {n_fc_at} unique values, expected {expected_fc_at_count}"
    )
    print(f"  ✓ forecast_created_at: {n_fc_at} unique values (task convention: {fc_at_strategy})")
    
    # Verify 2025-12-04 is absent from target_date (sanity check on the 364)
    if fc_at_strategy == "previous_day":
        n_dec_4 = (df["target_date"] == pd.Timestamp("2025-12-04")).sum()
        assert n_dec_4 == 0, f"  ✗ Found {n_dec_4} rows for missing date 2025-12-04"
        print(f"  ✓ No rows for 2025-12-04 (correctly excluded)")
    
    he_min, he_max = df["he"].min(), df["he"].max()
    assert he_min == 1 and he_max == 24, f"  ✗ HE range {he_min}-{he_max} != 1-24"
    print(f"  ✓ HE range: {he_min}-{he_max}")
    
    td_min, td_max = df["target_date"].min(), df["target_date"].max()
    print(f"  ✓ target_date range: {td_min.date()} to {td_max.date()}")
    
    print(f"  ✓ predict_pd: mean={df['predict_pd'].mean():.2f}, "
          f"median={df['predict_pd'].median():.2f}, "
          f"max={df['predict_pd'].max():.2f}")
    
    print(f"  ✓ Sample row (first): "
          f"{df.iloc[0]['bus_id']} | {df.iloc[0]['zone_id']} | "
          f"target={df.iloc[0]['target_date'].date()} HE{df.iloc[0]['he']} | "
          f"predict_pd={df.iloc[0]['predict_pd']:.2f}")
    
    summary_rows.append({
        "file": filename,
        "size_mb": round(size_mb, 1),
        "rows": len(df),
        "model_name": expected_model_name,
        "fc_at_count": n_fc_at,
        "mean_predict_pd": round(df["predict_pd"].mean(), 2),
    })
    
    del df
    gc.collect()

print("\n" + "=" * 80)
print("Summary of all 6 baseline forecast files")
print("=" * 80)
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

print(f"\nAll 6 files written to: {FORECASTS_DIR.resolve()}")
print(f"Total size: {sum(r['size_mb'] for r in summary_rows):.1f} MB")
print(f"Total rows: {sum(r['rows'] for r in summary_rows):,}")
print("\n✓ Notebook 03 complete — all baseline forecast files ready for evaluation in notebook 06")

Final verification — notebook 03 outputs

────────────────────────────────────────────────────────────────────────────────
Checking: forecast_prev_week_nextday.parquet
────────────────────────────────────────────────────────────────────────────────
  ✓ File present (78.5 MB)
  ✓ Schema: 7 columns in correct order
  ✓ Row count: 32,427,554
  ✓ No NaN values in predict_pd
  ✓ model_name: snaive_prev_week_nextday
  ✓ forecast_created_at: 364 unique values (task convention: previous_day)
  ✓ No rows for 2025-12-04 (correctly excluded)
  ✓ HE range: 1-24
  ✓ target_date range: 2025-01-01 to 2025-12-31
  ✓ predict_pd: mean=14.09, median=7.90, max=1176.89
  ✓ Sample row (first): 36POD_138KV_1 | FWES | target=2025-01-01 HE1 | predict_pd=22.92

────────────────────────────────────────────────────────────────────────────────
Checking: forecast_prev_63d_nextmonth.parquet
────────────────────────────────────────────────────────────────────────────────
  ✓ File present (76.7 MB)
  ✓ Schema: 7 colum